In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# --- CONFIGURACIÓN DE LA CONEXIÓN ---
DB_CONFIG = {
    'user': os.environ['ML_DB_USER'],
    'password': os.environ['ML_DB_PASSWORD'],
    'host': 'localhost',       # Servidor (tu PC)
    'port': '5432',
    'database': 'gemelo_datos'
}

# Creamos la cadena de conexión
connection_str = f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

try:
    # Conectamos
    engine = create_engine(connection_str)
    
    # --- LA QUERY CLAVE ---
    # Traemos las últimas 50 lecturas para ver la frecuencia
    query = """
    SELECT * FROM lecturas_sensores 
    ORDER BY fecha DESC
    LIMIT 100;
    """
    
    # Cargamos en un DataFrame de Pandas
    df = pd.read_sql(query, engine)
    
    # Mostramos la tabla bonita
    if not df.empty:
        print(f"✅ Conexión exitosa. Mostrando las últimas {len(df)} lecturas:")
        display(df) # Esto pinta la tabla visual en VS Code
    else:
        print("⚠️ Conexión correcta, pero la tabla está VACÍA.")
        print("Espera unos minutos a que n8n guarde los primeros datos.")

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# 1. Aseguramos que Pandas entienda que eso es una fecha
df['fecha'] = pd.to_datetime(df['fecha'])

# 2. Le decimos: "Oye, estos datos vienen en UTC (Universal)"
# Y luego: "Conviértelos a la hora de Madrid/Europa"
df['fecha_es'] = df['fecha'].dt.tz_localize('UTC').dt.tz_convert('Europe/Madrid')

# 3. Mostramos las dos columnas para comparar
print("Comprobación de hora:")
display(df[['fecha', 'fecha_es']].head())